# triangle-splatting :: Custom data :: duke statue

-----
- Conda env : [waikiki_statue](README.md#setup-a-conda-environment)
-----

### Check system

In [2]:
!nvidia-smi

Fri Oct  3 11:46:30 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 575.57.08              Driver Version: 575.57.08      CUDA Version: 12.9     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 2080 Ti     On  |   00000000:01:00.0  On |                  N/A |
| 26%   41C    P8             31W /  250W |     549MiB /  11264MiB |      3%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Download a video

In [3]:
import os
from pathlib import Path

Path("./temp_data").mkdir(exist_ok=True, parents=True)

In [4]:
import gdown
VIDEO_NAME = "duke_statue"

FPS = 10
RES = 4
id = "1fFvegLaEc_pGmputDUD6InIQFf8auEC1"
vid_path = f"./temp_data/{VIDEO_NAME}.mov"

gdown.download(id=id, output = vid_path)

Downloading...
From: https://drive.google.com/uc?id=1fFvegLaEc_pGmputDUD6InIQFf8auEC1
To: /home/hyunjae/110_HyunJae_Git/2025_Playgrounds/CV_Playgrounds/3DCV/triangle_splatting/temp_data/duke_statue.mov
100%|██████████| 11.5M/11.5M [00:01<00:00, 6.17MB/s]


'./temp_data/duke_statue.mov'

### Extract images from the video

In [5]:
DATASET_DIR_PATH = f"./temp_data/{VIDEO_NAME}"
IMAGES_DIR_PATH = os.path.join(DATASET_DIR_PATH, "images")
DATABASE_PATH = os.path.join(DATASET_DIR_PATH, "database.db")

Path(IMAGES_DIR_PATH).mkdir(exist_ok=True, parents=True)


!ffmpeg -i $vid_path -vf fps=$FPS $IMAGES_DIR_PATH/frame_%04d.jpg

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

### Colmap :: Feature Extraction

In [6]:
# Colmap Feature Extraction
!colmap feature_extractor \
    --database_path $DATABASE_PATH \
    --image_path $IMAGES_DIR_PATH  --ImageReader.camera_model PINHOLE


Feature extraction

Processed file [1/102]
  Name:            frame_0001.jpg
  Dimensions:      1920 x 1080
  Camera:          #1 - PINHOLE
  Focal Length:    2304.00px
  Features:        4388
Processed file [2/102]
  Name:            frame_0002.jpg
  Dimensions:      1920 x 1080
  Camera:          #2 - PINHOLE
  Focal Length:    2304.00px
  Features:        4349
Processed file [3/102]
  Name:            frame_0003.jpg
  Dimensions:      1920 x 1080
  Camera:          #3 - PINHOLE
  Focal Length:    2304.00px
  Features:        4328
Processed file [4/102]
  Name:            frame_0010.jpg
  Dimensions:      1920 x 1080
  Camera:          #10 - PINHOLE
  Focal Length:    2304.00px
  Features:        4339
Processed file [5/102]
  Name:            frame_0005.jpg
  Dimensions:      1920 x 1080
  Camera:          #5 - PINHOLE
  Focal Length:    2304.00px
  Features:        4446
Processed file [6/102]
  Name:            frame_0006.jpg
  Dimensions:      1920 x 1080
  Camera:          #6 - P

### Colmap :: Feature Matching

In [7]:
# Feature Matching
!colmap sequential_matcher \
    --database_path $DATABASE_PATH


Sequential feature matching

Matching image [1/102] in 0.305s
Matching image [2/102] in 0.282s
Matching image [3/102] in 0.364s
Matching image [4/102] in 0.332s
Matching image [5/102] in 0.328s
Matching image [6/102] in 0.095s
Matching image [7/102] in 0.175s
Matching image [8/102] in 0.258s
Matching image [9/102] in 0.276s
Matching image [10/102] in 0.300s
Matching image [11/102] in 0.325s
Matching image [12/102] in 0.108s
Matching image [13/102] in 0.394s
Matching image [14/102] in 0.245s
Matching image [15/102] in 0.348s
Matching image [16/102] in 0.265s
Matching image [17/102] in 0.414s
Matching image [18/102] in 0.302s
Matching image [19/102] in 0.314s
Matching image [20/102] in 0.388s
Matching image [21/102] in 0.233s
Matching image [22/102] in 0.100s
Matching image [23/102] in 0.187s
Matching image [24/102] in 0.447s
Matching image [25/102] in 0.282s
Matching image [26/102] in 0.115s
Matching image [27/102] in 0.212s
Matching image [28/102] in 0.200s
Matching image [29/102] in 

### Colmap :: Sparse Reconstruction (Mapper)

In [8]:
# Sparse Reconstruction (Mapper)
SPARSE_DIR = os.path.join(DATASET_DIR_PATH, "sparse")
Path(SPARSE_DIR).mkdir(exist_ok=True, parents=True)

!colmap mapper \
    --database_path $DATABASE_PATH \
    --image_path $IMAGES_DIR_PATH \
    --output_path $SPARSE_DIR


Loading database

Loading cameras... 102 in 0.000s
Loading matches... 1021 in 0.020s
Loading images... 102 in 0.023s (connected 102)
Building correspondence graph... in 0.095s (ignored 0)

Elapsed time: 0.002 [minutes]


Finding good initial image pair


Initializing with image pair #9 and #25


Global bundle adjustment

iter      cost      cost_change  |gradient|   |step|    tr_ratio  tr_radius  ls_iter  iter_time  total_time
   0  5.319652e+01    0.00e+00    1.01e+03   0.00e+00   0.00e+00  1.00e+04        0    1.06e-03    4.26e-03
   1  4.840260e+01    4.79e+00    8.65e+02   3.78e+01   9.99e-01  3.00e+04        1    2.72e-03    7.02e-03
   2  4.753984e+01    8.63e-01    1.56e+03   5.40e+01   8.91e-01  5.73e+04        1    5.57e-04    7.58e-03
   3  4.721590e+01    3.24e-01    3.67e+02   4.14e+01   7.95e-01  7.22e+04        1    5.48e-04    8.14e-03
   4  4.706028e+01    1.56e-01    1.49e+02   2.84e+01   7.95e-01  9.09e+04        1    5.51e-04    8.69e-03
   5  4.697081e+01    8.95e-

### Triangle-Splatting :: Training (Indoor mode)

In [9]:
DATASET_DIR_PATH
OUTPUR_DIR_PATH = f"./temp_result/{VIDEO_NAME}"
print(DATASET_DIR_PATH)
print(OUTPUR_DIR_PATH)

!CUDA_VISIBLE_DEVICES=0 python temp_triangle-splatting/train.py -s $DATASET_DIR_PATH -m $OUTPUR_DIR_PATH -r $RES --eval

./temp_data/duke_statue
./temp_result/duke_statue
Optimizing ./temp_result/duke_statue
Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]
/home/hyunjae/anaconda3/envs/triangle_splatting/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/hyunjae/anaconda3/envs/triangle_splatting/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Loading model from: /home/hyunjae/anaconda3/envs/triangle_splatting/lib/python3.11/site-packages/lpips/weights/v0.1/vgg.pth
Output folder: ./temp_resul

### Triangle-Splatting :: Rendering (Indoor mode)

In [10]:
!CUDA_VISIBLE_DEVICES=0 python temp_triangle-splatting/render.py -m $OUTPUR_DIR_PATH

Looking for config file in ./temp_result/duke_statue/cfg_args
Config file found: ./temp_result/duke_statue/cfg_args
Rendering ./temp_result/duke_statue
Loading trained model at iteration 30000 [03/10 12:05:59]
Reading camera 1/102----- PINHOLE [03/10 12:05:59]
Reading camera 2/102----- PINHOLE [03/10 12:05:59]
Reading camera 3/102----- PINHOLE [03/10 12:05:59]
Reading camera 4/102----- PINHOLE [03/10 12:05:59]
Reading camera 5/102----- PINHOLE [03/10 12:05:59]
Reading camera 6/102----- PINHOLE [03/10 12:05:59]
Reading camera 7/102----- PINHOLE [03/10 12:05:59]
Reading camera 8/102----- PINHOLE [03/10 12:05:59]
Reading camera 9/102----- PINHOLE [03/10 12:05:59]
Reading camera 10/102----- PINHOLE [03/10 12:05:59]
Reading camera 11/102----- PINHOLE [03/10 12:05:59]
Reading camera 12/102----- PINHOLE [03/10 12:05:59]
Reading camera 13/102----- PINHOLE [03/10 12:05:59]
Reading camera 14/102----- PINHOLE [03/10 12:05:59]
Reading camera 15/102----- PINHOLE [03/10 12:05:59]
Reading camera 16/1

### Triangle-Splatting :: Create a video (Indoor mode)

In [11]:
!CUDA_VISIBLE_DEVICES=0 python temp_triangle-splatting/create_video.py -m $OUTPUR_DIR_PATH

Looking for config file in ./temp_result/duke_statue/cfg_args
Config file found: ./temp_result/duke_statue/cfg_args
Creating video for ./temp_result/duke_statue
Loading trained model at iteration 30000
Reading camera 1/102----- PINHOLE
Reading camera 2/102----- PINHOLE
Reading camera 3/102----- PINHOLE
Reading camera 4/102----- PINHOLE
Reading camera 5/102----- PINHOLE
Reading camera 6/102----- PINHOLE
Reading camera 7/102----- PINHOLE
Reading camera 8/102----- PINHOLE
Reading camera 9/102----- PINHOLE
Reading camera 10/102----- PINHOLE
Reading camera 11/102----- PINHOLE
Reading camera 12/102----- PINHOLE
Reading camera 13/102----- PINHOLE
Reading camera 14/102----- PINHOLE
Reading camera 15/102----- PINHOLE
Reading camera 16/102----- PINHOLE
Reading camera 17/102----- PINHOLE
Reading camera 18/102----- PINHOLE
Reading camera 19/102----- PINHOLE
Reading camera 20/102----- PINHOLE
Reading camera 21/102----- PINHOLE
Reading camera 22/102----- PINHOLE
Reading camera 23/102----- PINHOLE
Re

### Triangle-Splatting :: Training (Outdoor mode)

In [12]:
DATASET_DIR_PATH
OUTDOOR_OUTPUR_DIR_PATH = f"./temp_result/{VIDEO_NAME}_outdoor"
print(DATASET_DIR_PATH)
print(OUTDOOR_OUTPUR_DIR_PATH)

!CUDA_VISIBLE_DEVICES=0 python temp_triangle-splatting/train.py -s $DATASET_DIR_PATH -m $OUTDOOR_OUTPUR_DIR_PATH -r $RES --eval  --outdoor 

./temp_data/duke_statue
./temp_result/duke_statue_outdoor
Optimizing ./temp_result/duke_statue_outdoor
Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]
/home/hyunjae/anaconda3/envs/triangle_splatting/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/hyunjae/anaconda3/envs/triangle_splatting/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Loading model from: /home/hyunjae/anaconda3/envs/triangle_splatting/lib/python3.11/site-packages/lpips/weights/v0.1/vgg.pth
Output fold

### Triangle-Splatting :: Rendering (Outdoor mode)

In [13]:
!CUDA_VISIBLE_DEVICES=0 python temp_triangle-splatting/render.py -m $OUTDOOR_OUTPUR_DIR_PATH

Looking for config file in ./temp_result/duke_statue_outdoor/cfg_args
Config file found: ./temp_result/duke_statue_outdoor/cfg_args
Rendering ./temp_result/duke_statue_outdoor
Loading trained model at iteration 30000 [03/10 12:24:52]
Reading camera 1/102----- PINHOLE [03/10 12:24:52]
Reading camera 2/102----- PINHOLE [03/10 12:24:52]
Reading camera 3/102----- PINHOLE [03/10 12:24:52]
Reading camera 4/102----- PINHOLE [03/10 12:24:52]
Reading camera 5/102----- PINHOLE [03/10 12:24:52]
Reading camera 6/102----- PINHOLE [03/10 12:24:52]
Reading camera 7/102----- PINHOLE [03/10 12:24:52]
Reading camera 8/102----- PINHOLE [03/10 12:24:52]
Reading camera 9/102----- PINHOLE [03/10 12:24:52]
Reading camera 10/102----- PINHOLE [03/10 12:24:52]
Reading camera 11/102----- PINHOLE [03/10 12:24:52]
Reading camera 12/102----- PINHOLE [03/10 12:24:52]
Reading camera 13/102----- PINHOLE [03/10 12:24:52]
Reading camera 14/102----- PINHOLE [03/10 12:24:52]
Reading camera 15/102----- PINHOLE [03/10 12:24

### Triangle-Splatting :: Create a video (Outdoor mode)

In [14]:
!CUDA_VISIBLE_DEVICES=0 python temp_triangle-splatting/create_video.py -m $OUTDOOR_OUTPUR_DIR_PATH

Looking for config file in ./temp_result/duke_statue_outdoor/cfg_args
Config file found: ./temp_result/duke_statue_outdoor/cfg_args
Creating video for ./temp_result/duke_statue_outdoor
Loading trained model at iteration 30000
Reading camera 1/102----- PINHOLE
Reading camera 2/102----- PINHOLE
Reading camera 3/102----- PINHOLE
Reading camera 4/102----- PINHOLE
Reading camera 5/102----- PINHOLE
Reading camera 6/102----- PINHOLE
Reading camera 7/102----- PINHOLE
Reading camera 8/102----- PINHOLE
Reading camera 9/102----- PINHOLE
Reading camera 10/102----- PINHOLE
Reading camera 11/102----- PINHOLE
Reading camera 12/102----- PINHOLE
Reading camera 13/102----- PINHOLE
Reading camera 14/102----- PINHOLE
Reading camera 15/102----- PINHOLE
Reading camera 16/102----- PINHOLE
Reading camera 17/102----- PINHOLE
Reading camera 18/102----- PINHOLE
Reading camera 19/102----- PINHOLE
Reading camera 20/102----- PINHOLE
Reading camera 21/102----- PINHOLE
Reading camera 22/102----- PINHOLE
Reading camer